# Hands-On 1: Build Your First LangGraph

In this notebook, we build a foundational LangGraph workflow step-by-step: State → Nodes → Edges → Compile → Invoke → Final State.

#### Imports

We import `TypedDict` to define the state schema and `StateGraph`, `START`, `END` from LangGraph to build the graph.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

#### Create Graph State

State is the shared data structure of the LangGraph workflow. It stores information that flows between nodes. Here we define an `AgentState` with four fields: `question`, `research`, `analysis`, and `answer`.

In [3]:
class AgentState(TypedDict):
    question: str
    research: str
    analysis: str
    answer: str

#### Create First Node

A Node is a Python function that performs a specific task. It reads the State and returns updated values. The `research_node` takes the question from state and produces a research string.

In [4]:
def research_node(state: AgentState):
    return {
        "research": f"Research completed for: {state['question']}"
    }

#### Create More Nodes

We need analysis and answer nodes so the workflow is a full pipeline. `analysis_node` processes the research, and `answer_node` generates the final answer.

In [5]:
def analysis_node(state: AgentState):
    return {
        "analysis": f"Analyzed findings: {state['research']}"
    }


def answer_node(state: AgentState):
    return {
        "answer": f"Answer generated for: {state['question']}"
    }

#### Add Nodes to Graph

`add_node()` registers Python functions as nodes in the graph. Each node represents a specific task in the workflow.

In [6]:
graph = StateGraph(AgentState)

graph.add_node("research", research_node)
graph.add_node("analysis", analysis_node)
graph.add_node("answer", answer_node)

#### Connect Nodes

Edges define the execution path between nodes. They determine which node runs next. We connect START → research → analysis → answer → END.

In [7]:
graph.add_edge(START, "research")
graph.add_edge("research", "analysis")
graph.add_edge("analysis", "answer")
graph.add_edge("answer", END)

#### Compile Graph

`compile()` converts the graph definition into an executable workflow.

In [8]:
agent = graph.compile()

#### Invoke Graph

`invoke()` starts the graph with an initial State. The graph then executes the connected nodes in order.

In [9]:
result = agent.invoke({
    "question": "What is RAG?",
    "research": "",
    "analysis": "",
    "answer": ""
})

#### Inspect Final State

The final State contains the values generated or updated by the nodes. We print the result to see the full output.

In [10]:
print(result)

{'question': 'What is RAG?', 'research': 'Research completed for: What is RAG?', 'analysis': 'Analyzed findings: Research completed for: What is RAG?', 'answer': 'Answer generated for: What is RAG?'}


#### Summary

This is the foundation for all LangGraph applications. We defined state, created nodes, connected them with edges, compiled the graph, and invoked it to get a final state.